# Подготовка данных: подвыборка Yambda-50M

Ноутбук скачивает [Yambda](https://huggingface.co/datasets/yandex/yambda) (Яндекс Музыка, Apache 2.0, [arXiv:2505.22238](https://arxiv.org/abs/2505.22238)),
смотрит на сырые данные и строит воспроизводимую подвыборку под ограничения ДЗ (100k–2M взаимодействий, ≥1000 пользователей и объектов).

**Запуск на Kaggle:** Settings → Internet **On**, Accelerator **None** (GPU здесь не нужен), затем Run All.
Логирование в ClearML (проект `yambda-recsys`) включается флагом `USE_CLEARML` и требует Kaggle Secrets `CLEARML_*`.
Результат лежит в `/kaggle/working/yambda_subsample/`; его сохраняем как Kaggle Dataset (Output → New Dataset)
и дальше подключаем к `experiments.ipynb`.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from huggingface_hub import hf_hub_download

print("polars", pl.__version__)

SEED = 42
REPO = "yandex/yambda"
ON_KAGGLE = os.path.exists("/kaggle/working")
RAW_DIR = "/kaggle/tmp/raw" if ON_KAGGLE else "data/raw"  # сырьё не кладём в output
OUT_DIR = "/kaggle/working/yambda_subsample" if ON_KAGGLE else "data/yambda_subsample"

DAY = 24 * 3600  # timestamp — секунды от начала лога, округлённые до 5 с
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
USE_CLEARML = False  # True — логировать запуск в ClearML (нужны Kaggle Secrets CLEARML_*)


class _Noop:
    def __getattr__(self, name):
        return lambda *args, **kwargs: None


task = clog = _Noop()
CLEARML_KEYS = ["CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST",
                "CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"]
if USE_CLEARML and ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        for k in CLEARML_KEYS:
            os.environ[k] = secrets.get_secret(k)
    except Exception as e:
        raise RuntimeError("Не найдены Kaggle Secrets CLEARML_*: Add-ons → Secrets, подключить к ноутбуку") from e

if USE_CLEARML:
    from clearml import Task
    task = Task.init(project_name="yambda-recsys", task_name="prepare-data/yambda-50m/v2",
                     task_type=Task.TaskTypes.data_processing, auto_connect_frameworks=False,
                     reuse_last_task_id=False)
    task.add_tags(["iter1", "kaggle" if ON_KAGGLE else "local", "cpu"])
    task.connect({"seed": SEED, "source": f"https://huggingface.co/datasets/{REPO} (flat/50m)"}, name="general")
    clog = task.get_logger()

## 1. Загрузка

Берём самую маленькую плоскую версию `flat/50m` (10 000 пользователей) и маппинги трек → артист / альбом.
`multi_event.parquet` не качаем: он дублирует отдельные файлы событий. Аудио-эмбеддинги (~13 GB) пока не трогаем.

In [ ]:
FILES = {
    "listens": "flat/50m/listens.parquet",
    "likes": "flat/50m/likes.parquet",
    "dislikes": "flat/50m/dislikes.parquet",
    "unlikes": "flat/50m/unlikes.parquet",
    "undislikes": "flat/50m/undislikes.parquet",
    "artist_map": "artist_item_mapping.parquet",
    "album_map": "album_item_mapping.parquet",
}
paths = {k: hf_hub_download(REPO, f, repo_type="dataset", local_dir=RAW_DIR) for k, f in FILES.items()}

listens = pl.read_parquet(paths["listens"])
feedback = {k: pl.read_parquet(paths[k]) for k in ["likes", "dislikes", "unlikes", "undislikes"]}
artist_map = pl.read_parquet(paths["artist_map"])
album_map = pl.read_parquet(paths["album_map"])

## 2. Что лежит в сырых данных

In [ ]:
def overview(name, df):
    row = {"file": name, "rows": df.height}
    if "uid" in df.columns:
        row |= {
            "users": df["uid"].n_unique(),
            "items": df["item_id"].n_unique(),
            "day_min": df["timestamp"].min() / DAY,
            "day_max": df["timestamp"].max() / DAY,
        }
    return row

raw_overview = pl.DataFrame([overview("listens", listens)] + [overview(k, v) for k, v in feedback.items()])
raw_overview.write_csv(f"{OUT_DIR}/raw_overview.csv")
clog.report_table("raw", "overview", iteration=0, table_plot=raw_overview.to_pandas())
raw_overview

In [ ]:
for name, df in [("listens", listens), *feedback.items(), ("artist_map", artist_map), ("album_map", album_map)]:
    print(f"--- {name}: {df.schema}")
    nulls = {c: n for c, n in zip(df.columns, df.null_count().row(0)) if n}
    print("    nulls:", nulls or "нет")
listens.head()

In [ ]:
# Полные дубликаты строк и дубликаты по ключу (uid, item_id, timestamp)
n_full_dup = listens.height - listens.unique().height
n_key_dup = listens.height - listens.unique(subset=["uid", "item_id", "timestamp"]).height
print(f"полных дубликатов: {n_full_dup:,}; дубликатов по (uid, item_id, timestamp): {n_key_dup:,}")
clog.report_single_value("raw_full_duplicates", n_full_dup)
clog.report_single_value("raw_key_duplicates", n_key_dup)

**Очистка дубликатов.** Полные дубликаты — артефакт логирования, их удаляем. Если у одного (uid, item_id, timestamp)
несколько записей с разными полями, это одно событие, записанное несколько раз: оставляем запись с максимальной
долей дослушивания. В событиях лайков и дизлайков удаляем полные дубликаты.

In [ ]:
n_before = listens.height
listens = (
    listens.unique(maintain_order=True)
    .sort(["uid", "timestamp", "item_id", "played_ratio_pct"], descending=[False, False, False, True])
    .unique(subset=["uid", "item_id", "timestamp"], keep="first", maintain_order=True)
)
feedback = {k: v.unique(maintain_order=True) for k, v in feedback.items()}
print(f"listens: {n_before:,} → {listens.height:,} (удалено {n_before - listens.height:,})")
clog.report_single_value("clean_listens", listens.height)

In [ ]:
# Маппинги: сколько треков покрыто и бывает ли у трека несколько артистов / альбомов
for name, m in [("artist_map", artist_map), ("album_map", album_map)]:
    per_item = m.group_by("item_id").len()
    covered = listens.select(pl.col("item_id").unique()).join(m, on="item_id", how="semi").height
    print(f"{name}: {m.height:,} строк, {per_item.height:,} треков, "
          f"треков с >1 значением: {(per_item['len'] > 1).sum():,}, "
          f"покрытие треков из listens: {covered / listens['item_id'].n_unique():.1%}")
    clog.report_single_value(f"{name}_multi_value_items", int((per_item['len'] > 1).sum()))
    clog.report_single_value(f"{name}_coverage", covered / listens['item_id'].n_unique())

In [ ]:
lp = listens["played_ratio_pct"]
print(f"Listen+ (played_ratio_pct > 50): {(lp > 50).mean():.1%}")
print(f"played_ratio_pct > 100 (перемотки): {(lp > 100).mean():.1%}")
print(f"органических прослушиваний: {listens['is_organic'].mean():.1%}")
clog.report_single_value("raw_listen_plus_share", (lp > 50).mean())
clog.report_single_value("raw_organic_share", listens["is_organic"].mean())

## 3. Активность во времени

Timestamp — секунды от начала лога, округлённые до 5 с (в README сказано «5s units», но все значения кратны 5,
а медианный интервал между соседними прослушиваниями ≈ медианной длине трека, ~3 минуты; лог покрывает ~300 дней, как в статье). Смотрим, сколько дней покрывает лог и равномерна ли активность:
от этого зависит, какое окно брать и как резать train / val / test.

In [ ]:
daily = (
    listens.with_columns((pl.col("timestamp") // DAY).alias("day"))
    .group_by("day")
    .agg(pl.len().alias("listens"), pl.col("uid").n_unique().alias("active_users"))
    .sort("day")
)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(daily["day"], daily["listens"])
axes[0].set_ylabel("прослушиваний в день")
axes[1].plot(daily["day"], daily["active_users"], color="tab:green")
axes[1].set_ylabel("активных пользователей")
axes[1].set_xlabel("день от начала лога")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/daily_activity.png", dpi=120)
plt.show()
daily.write_csv(f"{OUT_DIR}/daily_activity.csv")
clog.report_matplotlib_figure("eda", "daily_activity", figure=fig, iteration=0, report_image=True)
print(f"лог покрывает {daily['day'].max() - daily['day'].min() + 1} дней")

## 4. Сетка вариантов подвыборки

Процедура отбора (все шаги детерминированы, случайность только в выборе пользователей с `SEED`):

1. **Окно по времени:** последние `n_days` дней лога. Свежие данные ближе к реальному сценарию и дают естественное временное разбиение.
2. **Выбор пользователей:** доля `user_frac` пользователей, случайно с фиксированным seed.
3. **Взаимодействие** = пара (пользователь, трек) с хотя бы одним прослушиванием в окне; повторные прослушивания схлопываются.
4. **k-core:** итеративно удаляем треки, у которых меньше `min_item_users` слушателей, и пользователей, у которых меньше `min_user_items` треков, пока всё не стабилизируется.

Считаем статистику для сетки параметров, чтобы выбрать вариант, который попадает в 100k–2M взаимодействий и не слишком разрежен.

In [ ]:
def kcore(pairs, min_user_items, min_item_users):
    while True:
        before = pairs.height
        pairs = pairs.filter(pl.len().over("item_id") >= min_item_users)
        pairs = pairs.filter(pl.len().over("uid") >= min_user_items)
        if pairs.height == before:
            return pairs


def window(listens, n_days):
    t_max = listens["timestamp"].max()
    return listens.filter(pl.col("timestamp") > t_max - n_days * DAY)


def subsample(win, user_frac, min_item_users, min_user_items, seed=SEED):
    users = win["uid"].unique().sort()
    if user_frac < 1:
        users = users.sample(fraction=user_frac, seed=seed)
    win = win.filter(pl.col("uid").is_in(users.implode()))
    pairs = win.group_by("uid", "item_id").agg(pl.len().alias("n_listens"))
    pairs = kcore(pairs, min_user_items, min_item_users)
    events = win.join(pairs.select("uid", "item_id"), on=["uid", "item_id"], how="semi")
    return events, pairs


def stats(events, pairs):
    n_u, n_i = pairs["uid"].n_unique(), pairs["item_id"].n_unique()
    per_user = pairs.group_by("uid").len()["len"]
    return {
        "users": n_u,
        "items": n_i,
        "pairs": pairs.height,
        "listen_events": events.height,
        "density_%": round(100 * pairs.height / (n_u * n_i), 3),
        "median_items_per_user": int(per_user.median()),
    }

In [ ]:
GRID_DAYS = [30, 60, 90, 120]
GRID_USER_FRAC = [1.0, 0.5, 0.25]
GRID_MIN_ITEM_USERS = [20, 40, 80]
MIN_USER_ITEMS = 10

rows = []
for n_days in GRID_DAYS:
    win = window(listens, n_days)
    for user_frac in GRID_USER_FRAC:
        for min_item_users in GRID_MIN_ITEM_USERS:
            ev, pr = subsample(win, user_frac, min_item_users, MIN_USER_ITEMS)
            rows.append({"n_days": n_days, "user_frac": user_frac, "min_item_users": min_item_users,
                         **stats(ev, pr)})
grid = pl.DataFrame(rows)
grid.write_csv(f"{OUT_DIR}/subsample_grid.csv")
clog.report_table("subsample", "grid", iteration=0, table_plot=grid.to_pandas())
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(grid)

## 5. Выбранная подвыборка

Критерии выбора по сетке выше:

- ≤ 2M и по парам (пользователь, трек), и по сырым событиям прослушивания — ограничение задания выполняется при любом прочтении «взаимодействия»;
- ≥ 1000 пользователей и треков;
- окно ≥ 60 дней, чтобы после отрезания val и test (по неделе) в train осталось ~6 недель;
- среди подходящих вариантов — самый большой каталог, чтобы задача не вырождалась в выбор из пары тысяч хитов.

Подходят `60/0.25/20`, `60/0.25/40` и `90/0.25/80`; берём `60/0.25/20` (13k треков против 6k и 3k).

In [ ]:
CHOSEN = dict(n_days=60, user_frac=0.25, min_item_users=20, min_user_items=MIN_USER_ITEMS)
task.connect(CHOSEN, name="subsample")

win = window(listens, CHOSEN["n_days"])
sub_listens, sub_pairs = subsample(win, CHOSEN["user_frac"], CHOSEN["min_item_users"], CHOSEN["min_user_items"])
sub_stats = stats(sub_listens, sub_pairs)
sub_stats

In [ ]:
assert 100_000 <= sub_stats["pairs"] <= 2_000_000, sub_stats
assert sub_stats["users"] >= 1000 and sub_stats["items"] >= 1000, sub_stats

# Лайки / дизлайки и их отмены берём в том же окне и только по отобранным пользователям и трекам
t_from = listens["timestamp"].max() - CHOSEN["n_days"] * DAY
keep_users = sub_pairs["uid"].unique().implode()
keep_items = sub_pairs["item_id"].unique().implode()
sub_feedback = {
    k: v.filter((pl.col("timestamp") > t_from) & pl.col("uid").is_in(keep_users) & pl.col("item_id").is_in(keep_items))
    for k, v in feedback.items()
}
sub_artist = artist_map.filter(pl.col("item_id").is_in(keep_items))
sub_album = album_map.filter(pl.col("item_id").is_in(keep_items))

pl.DataFrame([{"file": k, "rows": v.height} for k, v in
              [("listens", sub_listens), *sub_feedback.items(), ("artist_map", sub_artist), ("album_map", sub_album)]])

In [ ]:
sub_listens.write_parquet(f"{OUT_DIR}/listens.parquet")
for k, v in sub_feedback.items():
    v.write_parquet(f"{OUT_DIR}/{k}.parquet")
sub_artist.write_parquet(f"{OUT_DIR}/artist_item_mapping.parquet")
sub_album.write_parquet(f"{OUT_DIR}/album_item_mapping.parquet")

meta = {
    "source": "https://huggingface.co/datasets/yandex/yambda (flat/50m)",
    "seed": SEED,
    "params": CHOSEN,
    "t_from": int(t_from),
    "t_max": int(listens["timestamp"].max()),
    "stats": sub_stats,
}
with open(f"{OUT_DIR}/meta.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))
for k, v in sub_stats.items():
    clog.report_single_value(k, v)
for name in ["meta.json", "raw_overview.csv", "subsample_grid.csv", "daily_activity.csv"]:
    task.upload_artifact(name, artifact_object=f"{OUT_DIR}/{name}")
task.close()
!ls -lh {OUT_DIR}